# IMDB Sentiment Analysis — RNN Approach

This notebook explores sentiment classification on the IMDB movie reviews dataset (50,000 reviews, balanced positive/negative) using a from-scratch RNN in PyTorch.

**Pipeline:**
1. Load and inspect the raw data, drop duplicates
2. Text preprocessing: lowercasing, URL/punctuation/HTML removal, stopword removal, stemming
3. Label encoding (positive/negative -> 1/0)
4. TF-IDF vectorization
5. Train/test split and PyTorch DataLoaders
6. A simple RNN classifier, trained and evaluated

**Note:** TF-IDF collapses each review into a single fixed-length vector, so there is no real sequence left for the RNN to process — the model here effectively behaves like a plain linear layer despite the RNN architecture. For a proper sequence model, raw tokenized text fed through an  layer would be the standard approach instead of TF-IDF. This notebook is kept as a practice exercise on the mechanics of PyTorch RNNs and DataLoaders rather than as the final modeling approach for this project.

In [33]:
import pandas as pd


In [34]:
df = pd.read_csv("IMDB Dataset.csv")

In [36]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [37]:
df.shape

(50000, 2)

In [38]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [39]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [40]:
df.drop_duplicates(inplace=True)

In [41]:
df.shape

(49582, 2)

# Preprocessing

## convorting to  lowercases

In [54]:
df["review"] = df["review"].str.lower()

## removing th urls

In [55]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)

In [56]:
new_text

'xyz is the word, xyz'

## removing punctualtions and additional symbols

In [57]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [58]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly s fmly lttle boy jke thks s zombe ...,negative
4,petter mtteis love time mey vully stunng ...,positive


## removing HTML

In [59]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [60]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly s fmly lttle boy jke thks s zombe ...,negative
4,petter mtteis love time mey vully stunng ...,positive


## removing of stopwords

In [61]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [62]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [63]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)

In [64]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [65]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,positive
1,wderful ltle prducti br br filming technique...,positive
2,hugh h werful w pen me h ummer weeken n...,positive
3,bcy fly e boy jke hk zobe cloe pn f...,negative
4,peer mei love i vu unng film wch mr mei...,positive


## stemming

In [67]:
# running -> run
# played -> play

from nltk.stem import PorterStemmer

In [69]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [70]:
df.head()

,review,sentiment
0,e revew nte wtchg 1 oz epo hook rght exctl hpp...,positive
1,wder ltle prducti br br film techniqu unssum l...,positive
2,hugh h wer w pen me h ummer weeken ng n r cne ...,positive
3,bci fli e boy jke hk zobe cloe pn fghg ebr br ...,negative
4,peer mei love i vu unng film wch mr mei fer u ...,positive


## encoding

In [71]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [72]:
y = df["sentiment"]

In [73]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

## vectorization

In [76]:
df.head()

,review,sentiment
0,e revew nte wtchg 1 oz epo hook rght exctl hpp...,1
1,wder ltle prducti br br film techniqu unssum l...,1
2,hugh h wer w pen me h ummer weeken ng n r cne ...,1
3,bci fli e boy jke hk zobe cloe pn fghg ebr br ...,0
4,peer mei love i vu unng film wch mr mei fer u ...,1


In [75]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [78]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3315433 stored elements and shape (49582, 5000)>

## dataset and data Loaders

In [80]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [101]:
X_train.shape

(39665, 5000)

In [102]:
X_test.shape

(9917, 5000)

In [104]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [106]:
X_train = X_train.toarray()
X_test = X_test.toarray()

AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [107]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [108]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)


## build RNN

In [110]:
import torch.nn as nn
import torch.optim as optimizer

In [116]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [118]:
import torch.optim as optim
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [121]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # direction =1
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.41915562748908997
epoch = 2/10 and loss = 0.42925527691841125
epoch = 3/10 and loss = 0.14589513838291168
epoch = 4/10 and loss = 0.17559771239757538
epoch = 5/10 and loss = 0.3681785762310028
epoch = 6/10 and loss = 0.15508224070072174
epoch = 7/10 and loss = 0.2650570273399353
epoch = 8/10 and loss = 0.24370183050632477
epoch = 9/10 and loss = 0.2529126703739166
epoch = 10/10 and loss = 0.2642979025840759


In [122]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 82.39386911364323
